# LX · 99 · Comparison across Exp 0-7 (saved outputs only)

**How to use:** *Runtime → Run all*. Nothing to edit. Allow Google Drive access when asked.

If Colab disconnects or the runtime ends, open the notebook again and *Run all* again: finished
work is kept on Google Drive and skipped, and the computation continues where it stopped.
Everything (checkpoints, results, figures, manifests) is written to Drive, never to the
temporary Colab disk.

**Suite:** D3 LCMV beamforming + Xu et al. (2025) protocol (methodology log entry 50, `docs/lcmv_xu.md`). All outputs
are under `<outputs>/D3_LCMV_XU/`; D1/D2 and the earlier D3 (wMNE) runs and results are never touched.
One row per mode x experiment x variant x setting: participant-level metrics (primary) and window-level metrics
(secondary), confusion counts, ROC curves, and the compute environment (CPU / GPU) of each result. Nothing is
trained, selected or ranked. Each run writes a new `reports/report_<time>/` folder.

In [ ]:
# ===== D3 LCMV / Xu suite setup: nothing to edit. Mounts Drive, gets the code, installs the pinned environment. =====
UPDATE_LX_CODE = False   # leave False. True only if you were asked to switch this suite to newer code
# The LCMV / Xu suite has its own code pin, <outputs>/D3_LCMV_XU/CODE_COMMIT.txt (all LX notebooks run
# that commit); the D1/D2 and D3 pins are only reported, never changed here.
BRANCH = 'claude/ecstatic-davinci-a2e8mh'
REPO_URL = 'https://github.com/Samimj9/Source-space-EEG-graph-learning-for-Alzheimer-s-disease-using-multi-band-functional-connectivity.git'

import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

def _run(cmd, cwd=None):  # shows output and stops "Run all" on failure
    p = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout)
    if p.returncode:
        raise RuntimeError(f'failed: {" ".join(cmd)}')

os.chdir('/content')
_run(['rm', '-rf', '/content/repo'])
_run(['git', 'clone', '-q', '-b', BRANCH, REPO_URL, '/content/repo'])
os.chdir('/content/repo')
_run([sys.executable, 'scripts/colab_setup.py', '--family', 'LX']
     + (['--update-code'] if UPDATE_LX_CODE else []))
sys.path.insert(0, '/content/repo/src')   # the package, for this notebook's own cells

from adgraph.colab import sh
from adgraph.config import load_paths
PATHS = load_paths('configs/paths.yaml')
OUT = PATHS['processed_dir']
print('outputs on Drive:', OUT)
print('Dataset 3 folder:', PATHS['dataset3']['root'])
LX = OUT + '/D3_LCMV_XU'
print('suite outputs:', LX)

## Comparison

In [ ]:
sh("python scripts/lx_report.py")
import glob
import pandas as pd
from IPython.display import Image, display
rep = sorted(glob.glob(LX + '/reports/report_*'))[-1]
pd.set_option('display.max_rows', 500, 'display.width', 250)
t = pd.read_csv(rep + '/comparison.csv')
if len(t):
    display(t[['mode', 'experiment', 'variant', 'setting', 'n_participants', 'participant_accuracy',
               'participant_balanced_accuracy', 'participant_sensitivity', 'participant_specificity',
               'participant_precision', 'participant_f1', 'participant_roc_auc', 'window_roc_auc',
               'window_balanced_accuracy', 'device']].round(3))
for png in sorted(glob.glob(rep + '/figures/*.png')):
    print(png)
    display(Image(filename=png))